# Step 1 — Data fetch & chart sanity check

이 노트북은 Binance 15분봉을 불러와 EMA200 차트를 확인하고, 신호 엔진이 어떤 정보를 생성하는지 빠르게 점검합니다.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(__file__).resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from data.binance_client import fetch_btc_eth_15m
from signals.ema_filter import ema_signal

plt.style.use("seaborn-v0_8")

In [ ]:
try:
    market_data = fetch_btc_eth_15m()
except RuntimeError as exc:
    print(f"Live fetch failed: {exc}\nUsing synthetic data for demonstration.")
    idx = pd.date_range(end=pd.Timestamp.utcnow(), periods=500, freq="15min")
    base = 20000 + np.cumsum(np.random.normal(0, 50, size=len(idx)))
    btc = pd.DataFrame(
        {
            "open": base,
            "high": base + np.random.uniform(10, 50, size=len(idx)),
            "low": base - np.random.uniform(10, 50, size=len(idx)),
            "close": base + np.random.uniform(-30, 30, size=len(idx)),
            "volume": np.random.uniform(100, 500, size=len(idx)),
        },
        index=idx,
    )
    eth = btc * 0.07
    market_data = {"BTCUSDT": btc, "ETHUSDT": eth}

for symbol, frame in market_data.items():
    display((symbol, frame.head(), frame.tail()))

In [ ]:
btc = market_data["BTCUSDT"].copy()
ema_result = ema_signal(btc)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(btc.index, btc["close"], label="Close", color="steelblue")
ax.plot(btc.index, ema_result.extra["ema"], label="EMA200", color="orange")
ax.set_title("BTCUSDT 15m close + EMA200")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Price")
ax.legend()
plt.tight_layout()
plt.show()

print(ema_result.summary)

In [ ]:
from main import compute_scores

results = compute_scores(market_data)
results